# ISA Mini-Project 1 – Sequential Playlist Recommender
## Spotify Million Playlist Dataset (MPD)

**Goal:** Given the tracks in a playlist (or a partial playlist), recommend the next track(s) a user would add.

This notebook covers:
1. Dataset overview and schema understanding
2. Exploratory Data Analysis (EDA)
3. Quest / problem formulation justified by the data
4. Data preparation for modeling

---
## 0. Setup

In [ ]:
import os, json, re, glob, random, collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from itertools import islice

OUT = Path('outputs')
OUT.mkdir(exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# ── CHANGE THIS to your local MPD path ──────────────────────────────────────
MPD_PATH = Path(r'../data')   # folder containing mpd.slice.*.json files
# ────────────────────────────────────────────────────────────────────────────

QUICK = True          # set False to process all 1 000 slices (slow)
MAX_SLICES = 10       # slices used when QUICK=True  (10 × 1000 playlists = 10 k)

print('MPD path:', MPD_PATH.resolve())
slices = sorted(MPD_PATH.glob('mpd.slice.*.json'))
print(f'Slices found: {len(slices)}')

---
## 1. Dataset Schema

Each JSON slice contains:
```
{
  "info": { "generated_on", "slice", "version" },
  "playlists": [
    {
      "pid", "name", "collaborative", "modified_at",
      "num_albums", "num_tracks", "num_followers",
      "num_edits", "duration_ms", "num_artists",
      "description" (optional),
      "tracks": [
        { "pos", "track_uri", "track_name",
          "artist_uri", "artist_name",
          "album_uri",  "album_name", "duration_ms" }
      ]
    }, ...
  ]
}
```
1 000 slices × 1 000 playlists = **1 000 000 playlists**, ~66 M track entries.

In [ ]:
# Peek at one slice
with open(slices[0]) as f:
    sample_slice = json.load(f)

print('Info:', sample_slice['info'])
pl = sample_slice['playlists'][0]
print('\nPlaylist fields:', list(pl.keys()))
print('Track fields:   ', list(pl['tracks'][0].keys()))
print(f'\nExample playlist: "{pl["name"]}"  ({pl["num_tracks"]} tracks)')
pd.DataFrame(pl['tracks'][:5])

---
## 2. Loading a Working Sample

In [ ]:
playlists = []
track_rows = []

use_slices = slices[:MAX_SLICES] if QUICK else slices

for path in use_slices:
    with open(path) as f:
        s = json.load(f)
    for pl in s['playlists']:
        playlists.append({
            'pid':          pl['pid'],
            'name':         pl['name'],
            'num_tracks':   pl['num_tracks'],
            'num_artists':  pl['num_artists'],
            'num_albums':   pl['num_albums'],
            'num_followers':pl['num_followers'],
            'num_edits':    pl['num_edits'],
            'duration_ms':  pl['duration_ms'],
            'modified_at':  pl['modified_at'],
            'has_desc':     'description' in pl,
            'collaborative':pl['collaborative'],
        })
        for t in pl['tracks']:
            track_rows.append({
                'pid':        pl['pid'],
                'pos':        t['pos'],
                'track_uri':  t['track_uri'],
                'track_name': t['track_name'],
                'artist_uri': t['artist_uri'],
                'artist_name':t['artist_name'],
                'album_uri':  t['album_uri'],
                'album_name': t['album_name'],
                'duration_ms':t['duration_ms'],
            })

pls  = pd.DataFrame(playlists)
trks = pd.DataFrame(track_rows)

print(f'Playlists : {len(pls):>8,}')
print(f'Track rows: {len(trks):>8,}')
pls.head(3)

---
## 3. Exploratory Data Analysis

### 3.1 Playlist-level statistics

In [ ]:
pls[['num_tracks','num_artists','num_albums','num_followers','num_edits']].describe().round(2)

| Column | Description |
|---|---|
| `num_tracks` | Number of tracks in the playlist (min 5, max 250 per dataset spec). Mean ~67, heavily right-skewed — most playlists are short, some are maxed out. |
| `num_artists` | Number of unique artists across all tracks. Correlates with `num_tracks`; mean ~38 suggests playlists mix ~1.8 tracks per artist on average. |
| `num_albums` | Number of unique albums. Close to `num_artists`, meaning most artists contribute from a single album per playlist. |
| `num_followers` | Number of Spotify users following the playlist. Extremely skewed (mean 3.14, max 11 745, but median 1) — the vast majority of playlists are private or have a single follower. |
| `num_edits` | Number of times the playlist was edited. Mean ~18, median 10 — most playlists were built incrementally over several sessions. |

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Playlist-level Distributions', fontsize=14, fontweight='bold')

cols = [
    ('num_tracks',    'Tracks per Playlist',    50),
    ('num_artists',   'Artists per Playlist',   50),
    ('num_albums',    'Albums per Playlist',    50),
    ('num_followers', 'Followers (log scale)',  50),
    ('num_edits',     'Edits per Playlist',     50),
    ('duration_ms',   'Duration (min)',         50),
]

for ax, (col, title, bins) in zip(axes.flat, cols):
    data = pls[col]
    if col == 'duration_ms':
        data = data / 60000
    ax.hist(data, bins=bins, color='steelblue', edgecolor='white', linewidth=0.3)
    ax.set_title(title)
    ax.set_xlabel(col if col != 'duration_ms' else 'minutes')
    ax.set_ylabel('count')
    if col == 'num_followers':
        ax.set_yscale('log')

plt.tight_layout()
plt.savefig(OUT / 'eda_playlist_distributions.png', bbox_inches='tight')
plt.show()

## Playlist-level Distributions

A 2×3 grid of histograms describing key playlist attributes.

| Plot | Description |
|---|---|
| **Tracks per Playlist** | Shows how many tracks each playlist contains. |
| **Artists per Playlist** | Shows how many unique artists appear in each playlist. |
| **Albums per Playlist** | Shows how many unique albums appear in each playlist. |
| **Followers (log scale)** | Shows how many Spotify users follow each playlist, plotted on a log y-axis. |
| **Edits per Playlist** | Shows how many times each playlist has been edited. |
| **Duration (min)** | Shows the total playback duration of each playlist in minutes. |

In [ ]:
print('Playlists with description:', pls['has_desc'].sum(), 
      f"({pls['has_desc'].mean()*100:.1f}%)")
print('Collaborative playlists:  ', (pls['collaborative']=='true').sum(),
      f"({(pls['collaborative']=='true').mean()*100:.2f}%)")

### 3.2 Track popularity (power-law distribution)

In [ ]:
track_counts = trks['track_uri'].value_counts()
artist_counts = trks['artist_uri'].value_counts()

print(f'Unique tracks : {len(track_counts):>8,}')
print(f'Unique artists: {len(artist_counts):>8,}')
print(f'Unique albums : {trks["album_uri"].nunique():>8,}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, counts, label in [
    (axes[0], track_counts,  'Track'),
    (axes[1], artist_counts, 'Artist'),
]:
    ax.loglog(range(1, len(counts)+1), counts.values, '.', markersize=1.5, alpha=0.5)
    ax.set_title(f'{label} Popularity (log-log)')
    ax.set_xlabel('Rank')
    ax.set_ylabel('Occurrences in playlists')

plt.tight_layout()
plt.savefig(OUT / 'eda_popularity_powerlaw.png', bbox_inches='tight')
plt.show()

# Long tail analysis
for pct in [1, 5, 10, 50]:
    n = int(len(track_counts) * pct / 100)
    cov = track_counts.iloc[:n].sum() / track_counts.sum() * 100
    print(f'Top {pct:3d}% of tracks cover {cov:.1f}% of all playlist entries')

### 3.3 Top tracks and artists

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

top_tracks = (
    trks.groupby(['track_uri','track_name','artist_name'])
    .size().reset_index(name='count')
    .nlargest(15, 'count')
)
top_tracks['label'] = top_tracks['track_name'].str[:30] + ' – ' + top_tracks['artist_name'].str[:20]

axes[0].barh(top_tracks['label'][::-1], top_tracks['count'][::-1], color='steelblue')
axes[0].set_title('Top 15 Tracks by Playlist Appearances')
axes[0].set_xlabel('Playlist count')

top_artists = (
    trks.groupby(['artist_uri','artist_name'])
    .size().reset_index(name='count')
    .nlargest(15, 'count')
)
axes[1].barh(top_artists['artist_name'][::-1], top_artists['count'][::-1], color='coral')
axes[1].set_title('Top 15 Artists by Track Entries')
axes[1].set_xlabel('Track entries')

plt.tight_layout()
plt.savefig(OUT / 'eda_top_tracks_artists.png', bbox_inches='tight')
plt.show()

### 3.4 Playlist name analysis

In [ ]:
def normalize_name(name):
    name = name.lower()
    name = re.sub(r"[.,\/#!$%\^\*;:{}=\_`~()@]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name

pls['name_norm'] = pls['name'].apply(normalize_name)
top_names = pls['name_norm'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top_names.index[::-1], top_names.values[::-1], color='mediumpurple')
ax.set_title('Top 20 Normalized Playlist Names')
ax.set_xlabel('Count')
plt.tight_layout()
plt.savefig(OUT / 'eda_playlist_names.png', bbox_inches='tight')
plt.show()

### 3.5 Track position analysis (sequential structure)

In [ ]:
# Which tracks appear most often at position 0 (opener tracks)?
openers = (
    trks[trks['pos'] == 0]
    .groupby(['track_uri', 'track_name', 'artist_name'])
    .size().reset_index(name='count')
    .nlargest(10, 'count')
)
openers['label'] = openers['track_name'].str[:35] + ' – ' + openers['artist_name'].str[:20]
print('Most common OPENER tracks (position 0):')
print(openers[['label','count']].to_string(index=False))

print()
# Do popular tracks appear more at the beginning or spread throughout?
top10_uris = track_counts.head(10).index.tolist()
pos_data = trks[trks['track_uri'].isin(top10_uris)].copy()
pos_data['track_short'] = pos_data['track_name'].str[:25]

fig, ax = plt.subplots(figsize=(12, 4))
for uri in top10_uris[:5]:
    subset = pos_data[pos_data['track_uri']==uri]
    label = subset['track_short'].iloc[0]
    ax.hist(subset['pos'], bins=50, alpha=0.5, label=label, density=True)
ax.set_title('Position Distribution of Top-5 Tracks')
ax.set_xlabel('Position in playlist')
ax.set_ylabel('Density')
ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig(OUT / 'eda_position_distribution.png', bbox_inches='tight')
plt.show()

### 3.6 Co-occurrence analysis (track-track)

In [ ]:
# For the top N tracks, count how often each pair appears in the same playlist
TOP_N = 50
top_n_uris = set(track_counts.head(TOP_N).index)

cooc = collections.Counter()
for pid, group in trks[trks['track_uri'].isin(top_n_uris)].groupby('pid'):
    uris = list(group['track_uri'].unique())
    for i in range(len(uris)):
        for j in range(i+1, len(uris)):
            pair = tuple(sorted([uris[i], uris[j]]))
            cooc[pair] += 1

print('Top 10 co-occurring track pairs (among top-50 tracks):')
uri_to_name = trks.drop_duplicates('track_uri').set_index('track_uri')['track_name']
for (a, b), cnt in cooc.most_common(10):
    print(f"  {cnt:5d}  {uri_to_name.get(a,'?')[:30]:30s}  <->  {uri_to_name.get(b,'?')[:30]}")

---
## 4. Quest Formulation

Based on the EDA findings, we define the recommendation task and justify design choices:

| Finding | Implication |
|---|---|
| Power-law track distribution: top 1% covers ~40% of entries | Popularity baseline is a strong signal; rare items need special handling |
| ~65 tracks/playlist on average, ordered by `pos` | Sequential (next-track) recommendation is natural |
| Strong co-occurrence signal in top tracks | Collaborative filtering / association rules are viable |
| Playlist names cluster into themes (workout, chill, rap…) | Name embedding can improve cold-start |
| Only ~20% of playlists have descriptions | Descriptions are an optional auxiliary feature |

**Task definition:** 
> Given the first *k* tracks of a playlist (seed), predict the next *m* tracks a user would add (continuation).  
> Evaluation metric: **R-Precision** and **NDCG@500**.

**Chosen approach progression:**
1. **Baseline** – Popularity-weighted recommendation (always recommend global top-N tracks not already in seed)
2. **Item-item CF** – Track co-occurrence matrix → cosine similarity nearest-neighbours
3. **Matrix Factorization** – Implicit ALS on playlist × track interaction matrix

---
## 5. Data Preparation for Modeling

### 5.1 Build integer ID mappings

In [ ]:
# Assign dense integer IDs to tracks and playlists
all_tracks  = trks['track_uri'].unique()
all_pids    = trks['pid'].unique()

track2id = {uri: i for i, uri in enumerate(all_tracks)}
id2track  = {i: uri for uri, i in track2id.items()}
pid2id   = {pid: i for i, pid in enumerate(all_pids)}

trks['track_id'] = trks['track_uri'].map(track2id)
trks['pl_id']    = trks['pid'].map(pid2id)

print(f'Unique tracks : {len(track2id):,}')
print(f'Unique pids   : {len(pid2id):,}')
trks[['pid','pl_id','pos','track_uri','track_id']].head()

### 5.2 Filter cold items (min support threshold)

In [ ]:
MIN_TRACK_SUPPORT = 5   # track must appear in at least 5 playlists
MIN_PL_LENGTH     = 5   # playlist must have at least 5 tracks (mirrors dataset spec)

track_support = trks['track_uri'].value_counts()
valid_tracks  = set(track_support[track_support >= MIN_TRACK_SUPPORT].index)

pl_lengths = trks.groupby('pid')['track_uri'].count()
valid_pids = set(pl_lengths[pl_lengths >= MIN_PL_LENGTH].index)

trks_filtered = trks[
    trks['track_uri'].isin(valid_tracks) &
    trks['pid'].isin(valid_pids)
].copy()

print(f'Tracks before/after filter: {len(track_support):,} → {len(valid_tracks):,}')
print(f'Rows   before/after filter: {len(trks):,} → {len(trks_filtered):,}')

# Proportion of data retained
print(f'Rows retained: {len(trks_filtered)/len(trks)*100:.1f}%')

### 5.3 Train / Validation / Test split

We split at the **playlist level** (not track level) to avoid data leakage.  
Within each test playlist we hold out the last *h* tracks as ground truth.

In [ ]:
random.seed(42)

all_pids_filtered = list(trks_filtered['pid'].unique())
random.shuffle(all_pids_filtered)

n = len(all_pids_filtered)
n_train = int(0.80 * n)
n_val   = int(0.10 * n)

train_pids = set(all_pids_filtered[:n_train])
val_pids   = set(all_pids_filtered[n_train:n_train+n_val])
test_pids  = set(all_pids_filtered[n_train+n_val:])

print(f'Train playlists: {len(train_pids):,}')
print(f'Val   playlists: {len(val_pids):,}')
print(f'Test  playlists: {len(test_pids):,}')

train_df = trks_filtered[trks_filtered['pid'].isin(train_pids)].copy()
val_df   = trks_filtered[trks_filtered['pid'].isin(val_pids)].copy()
test_df  = trks_filtered[trks_filtered['pid'].isin(test_pids)].copy()

### 5.4 Create seed / holdout sets for evaluation

The test playlists are split into two parts using a `HOLDOUT_FRAC` of 0.2:

- **Seed** – the first 80% of tracks (by position), used as input to the recommender
- **Holdout** – the last 20% of tracks, used as ground truth to evaluate recommendations against

For example, for a playlist of 50 tracks, the model sees tracks 0–39 and has to predict tracks 40–49.

In [ ]:
HOLDOUT_FRAC = 0.2   # hold out last 20% of each test playlist

def split_playlist(group):
    group = group.sort_values('pos')
    n = len(group)
    seed_n = max(1, int(n * (1 - HOLDOUT_FRAC)))
    seed    = group.iloc[:seed_n]
    holdout = group.iloc[seed_n:]
    return seed, holdout

test_seeds    = []
test_holdouts = []

for pid, group in test_df.groupby('pid'):
    seed, holdout = split_playlist(group)
    test_seeds.append(seed)
    test_holdouts.append({'pid': pid, 'holdout_tracks': list(holdout['track_uri'])})

test_seed_df = pd.concat(test_seeds)
test_holdout_df = pd.DataFrame(test_holdouts)

avg_seed    = test_seed_df.groupby('pid').size().mean()
avg_holdout = test_holdout_df['holdout_tracks'].apply(len).mean()
print(f'Avg seed length   : {avg_seed:.1f} tracks')
print(f'Avg holdout length: {avg_holdout:.1f} tracks')
test_holdout_df.head()

### 5.5 Build playlist-track interaction matrix (sparse)

Used for collaborative filtering and matrix factorization.

In [ ]:
from scipy.sparse import csr_matrix

# Re-index only on training data for clean IDs
train_tracks = list(train_df['track_uri'].unique())
train_pids_list = list(train_df['pid'].unique())

t2i = {t: i for i, t in enumerate(train_tracks)}
p2i = {p: i for i, p in enumerate(train_pids_list)}

rows = train_df['pid'].map(p2i)
cols = train_df['track_uri'].map(t2i)
# Binary (1 = track appears in playlist)
data = np.ones(len(train_df))

interaction_matrix = csr_matrix(
    (data, (rows, cols)),
    shape=(len(train_pids_list), len(train_tracks))
)

print(f'Interaction matrix shape: {interaction_matrix.shape}')
print(f'Density: {interaction_matrix.nnz / (interaction_matrix.shape[0]*interaction_matrix.shape[1]):.5f}')
print(f'NNZ: {interaction_matrix.nnz:,}')

In [ ]:
# Each playlist becomes a sequence of track URIs ordered by position
track_sequences = (
    train_df.sort_values('pos')
    .groupby('pid')['track_uri']
    .apply(list)
    .tolist()
)

print(f'Total sequences: {len(track_sequences):,}')
lengths = [len(s) for s in track_sequences]
print(f'Sequence length – mean: {np.mean(lengths):.1f}, median: {np.median(lengths):.0f}, max: {max(lengths)}')
print(f'\nExample sequence (first 5 tracks of playlist 0):')
print(track_sequences[0][:5])

### 5.6 Popularity baseline – sanity check

In [ ]:
# Global track popularity on training set
global_pop = train_df['track_uri'].value_counts()
top500_pop = global_pop.head(500).index.tolist()

def recommend_popular(seed_tracks, top_tracks, n=500):
    seed_set = set(seed_tracks)
    return [t for t in top_tracks if t not in seed_set][:n]

def r_precision(recommended, holdout):
    holdout_set = set(holdout)
    r = len(holdout_set)
    hits = sum(1 for t in recommended[:r] if t in holdout_set)
    return hits / r if r > 0 else 0.0

scores = []
for _, row in test_holdout_df.iterrows():
    pid = row['pid']
    seed = list(test_seed_df[test_seed_df['pid'] == pid]['track_uri'])
    recs = recommend_popular(seed, top500_pop)
    scores.append(r_precision(recs, row['holdout_tracks']))

print(f'Popularity baseline R-Precision: {np.mean(scores):.4f}')
print(f'(Evaluated on {len(scores)} test playlists)')